# API 로그 데이터 다루기

우리 서비스가 남긴 API 요청 기록을 판다스로 다듬고, 지표를 뽑습니다.

`api_logs.csv` 에는 **바로 쓸 수 없는 것들이 섞여 있습니다.** 그것을 찾아내 걸러내는 것이 앞부분이고, 남은 데이터로 숫자를 뽑는 것이 뒷부분입니다.

셀을 위에서부터 하나씩 실행합니다 (`Shift + Enter`).
`# TODO` 를 채우기 전에는 결과가 비어 있게 나옵니다. 고장난 것이 아니라 아직 안 채운 것입니다.

## 0. 로그 살펴보기

In [ ]:
import pandas as pd

df = pd.read_csv("api_logs.csv")
원본 = df.copy()          # 정제 전후를 비교하려고 남겨둔다

print("행, 열:", df.shape)
df.head()

어떤 요청이 얼마나 왔는지 먼저 봅니다.

In [ ]:
print(df["method"].value_counts().to_string())
print()
print(df["status_code"].value_counts().to_string())

각 열이 무엇인지 봅니다.

| 열 | 내용 |
| --- | --- |
| `ts` | 요청 시각 |
| `method` | `GET` · `POST` |
| `path` | 요청한 주소 |
| `status_code` | `200` 성공, `404`·`500` 실패 |
| `duration_ms` | 응답에 걸린 시간 (밀리초) |
| `user_id` | 로그인한 사용자. 없으면 비어 있음 |
| `user_agent` | 요청을 보낸 프로그램 |

In [ ]:
df.dtypes

### 비어 있는 값이 얼마나 되나

`isna()` 는 값이 비었으면 `True` 입니다. `sum()` 으로 세면 열마다 몇 개인지 나옵니다.

In [ ]:
df.isna().sum()

`duration_ms` 와 `user_id` 에 빈 값이 있습니다. **둘은 성격이 다릅니다.**

- `duration_ms` 가 없으면 응답시간을 계산할 수 없습니다
- `user_id` 가 없는 것은 **로그인하지 않고 보낸 요청**입니다. 정상이므로 버리면 안 됩니다

## 1. 정제

### 1-1. 중복 제거

같은 요청이 두 번 기록된 것이 있습니다. 사용자가 실패해서 다시 눌렀거나 프로그램이 재시도한 경우입니다.

In [ ]:
print("중복된 행:", df.duplicated().sum())

# TODO 1. 중복된 행을 지운다.  힌트: df.drop_duplicates()
df = ...

print("남은 행:", len(df))

### 1-2. 테스트성 트래픽 제거

사람이 보낸 요청이 아닌 것이 섞여 있습니다.

| 무엇 | 어떻게 알아보나 |
| --- | --- |
| 헬스체크 | `path` 가 `/health` |
| 확인용 스크립트 | `user_agent` 가 `python-httpx` 로 시작 |

이것들을 지표에 넣으면 요청 수가 부풀고 응답시간이 실제보다 짧게 나옵니다.

In [ ]:
print("헬스체크:", (df["path"] == "/health").sum())
print("스크립트:", df["user_agent"].str.startswith("python-httpx").sum())

# TODO 2. path 가 /health 인 행을 뺀다.  힌트: df[df["path"] != "/health"]
df = ...

# TODO 3. user_agent 가 python-httpx 로 시작하는 행을 뺀다.
#         힌트: 앞에 ~ 를 붙이면 "아닌 것" 이 된다
df = ...

print("남은 행:", len(df))

### 1-3. 결측 처리

`duration_ms` 가 없는 행은 응답시간 계산에 쓸 수 없으므로 뺍니다.
`user_id` 는 그대로 둡니다. 비로그인 요청도 사용량에 들어가야 합니다.

In [ ]:
print("duration_ms 없음:", df["duration_ms"].isna().sum())
print("user_id 없음:", df["user_id"].isna().sum(), "  (이건 그대로 둔다)")

# TODO 4. duration_ms 가 비어 있는 행만 뺀다.
#         힌트: df.dropna(subset=["duration_ms"])
df = ...

print("남은 행:", len(df))

### 1-4. 엔드포인트 표기 통일

`path` 에는 대화 번호가 그대로 들어 있습니다. 먼저 몇 종류인지 세어봅니다.

In [ ]:
print("path 종류:", df["path"].nunique())
df["path"].value_counts().head()

**대화마다 주소가 다르니 종류가 수십 개입니다.** 이대로 집계하면 "어느 기능이 많이 쓰였나" 를 알 수 없습니다.

번호 자리를 `{id}` 로 바꿔 같은 기능끼리 묶습니다.

In [ ]:
# TODO 5. path 의 번호 자리를 {id} 로 바꿔 endpoint 열을 만든다.
#         힌트: df["path"].str.replace(r"/[0-9a-f-]{36}", "/{id}", regex=True)
df["endpoint"] = ...

print("endpoint 종류:", df["endpoint"].nunique())
df["endpoint"].value_counts()

수십 종류가 네 종류로 줄었습니다. 이제 기능별로 집계할 수 있습니다.

### 1-5. 정제 전후 비교

무엇이 얼마나 달라졌는지 한 번에 봅니다.

In [ ]:
비교 = pd.DataFrame({
    "정제 전": [len(원본), 원본["path"].nunique(), round(원본["duration_ms"].mean(), 1)],
    "정제 후": [len(df), df["endpoint"].nunique(), round(df["duration_ms"].mean(), 1)],
}, index=["행 수", "주소 종류", "평균 응답시간(ms)"])

비교

**평균 응답시간이 크게 달라졌습니다.** 헬스체크는 2~9ms 로 아주 빨라서, 빼기 전에는 평균이 실제보다 짧게 나왔습니다.

## 2. 지표

### 2-1. 전체 요약

세 가지를 봅니다 — 얼마나 썼나, 얼마나 빨랐나, 얼마나 실패했나.

In [ ]:
print("전체 요청 수 :", len(df))
print(f"평균 응답시간 : {df['duration_ms'].mean():.1f} ms")

# TODO 6. status_code 가 400 이상인 비율을 백분율로 출력한다.
#         힌트: (df["status_code"] >= 400).mean() * 100
print(f"에러율        : ...")

`describe()` 로 응답시간의 분포를 한 번에 봅니다.

In [ ]:
df["duration_ms"].describe()

### 2-2. 엔드포인트별

`groupby` 로 기능별로 묶어 봅니다.

In [ ]:
df.groupby("endpoint")["duration_ms"].mean().round(1)

### 2-3. 느린 요청 찾기

평균은 하나의 숫자라 어느 요청이 느렸는지는 알려주지 않습니다. 정렬해서 직접 봅니다.

In [ ]:
# TODO 7. 응답시간이 긴 순서로 정렬해 위 5개를 본다.
#         힌트: df.sort_values("duration_ms", ascending=False).head(5)
...[["ts", "endpoint", "status_code", "duration_ms"]]

위쪽이 전부 `500` 이면 **실패한 요청이 느리다**는 뜻입니다. 상태 코드별 응답시간을 보면 확인됩니다.

In [ ]:
df.groupby("status_code")["duration_ms"].agg(["count", "mean"]).round(1)

### 2-4. 언제 많이 쓰나

`ts` 는 글자라서 그대로는 날짜로 묶을 수 없습니다. 날짜 형으로 바꾼 뒤 필요한 부분을 꺼냅니다.

In [ ]:
df["시각"] = pd.to_datetime(df["ts"])
df["날짜"] = df["시각"].dt.date

df["날짜"].value_counts().sort_index()

시간대별로도 봅니다. `.dt.hour` 로 시(hour)만 꺼냅니다.

In [ ]:
# TODO 8. 시간대별 요청 수를 센다.  힌트: df["시각"].dt.hour
...value_counts().sort_index()

### 2-5. 사용자별 요청 수

앞에서 `user_id` 가 빈 행을 남겨두었습니다. 여기서 그 이유가 드러납니다.

In [ ]:
print("로그인한 요청 :", df["user_id"].notna().sum())
print("비로그인 요청 :", df["user_id"].isna().sum())
print()
print("사용자별 요청 수")
print(df["user_id"].value_counts().to_string())

**비로그인 요청도 사용량입니다.** 앞에서 `dropna()` 를 그냥 썼다면 이 숫자가 통째로 사라졌을 겁니다.

In [ ]:
df.groupby("endpoint")["user_id"].apply(lambda s: s.isna().mean() * 100).round(1)

## 3. 저장

다음 시간에 이 파일을 읽어 화면에 띄웁니다.

In [ ]:
df.to_csv("clean_logs.csv", index=False)

print("clean_logs.csv 저장:", len(df), "행")